In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from spatial import find_municipality_udf


In [0]:
bronze_tomtom_flow = spark.readStream.table("bg_traffic.bg_traffic_bronze.tomtom_flow")


In [0]:
bronze_tomtom_flow.printSchema()

In [0]:
roads = ["autokomanda","brankov_most","bulevar_kralja_aleksandra","bulevar_mihajla_pupina","gazela","jurija_gagarina", "kneza_milosa", "most_na_adi","nemanjina", "omladinskih_brigada", "pancevacki_most", "savska", "takovska", "ustanicka", "vojislava_ilica"]

tomtom_flow_checkpoint = "/Volumes/bg_traffic/bg_traffic_silver/checkpoints/tomtom_flow"


### Flattening

In [0]:
def flatten_road(df, road):
    return df.select(
        F.lit(road).alias("road_name"),
        F.col("fetched_at"),
        F.col(f"data.{road}.flowSegmentData.currentSpeed").alias("current_speed"),
        F.col(f"data.{road}.flowSegmentData.currentTravelTime").alias("current_travel_time"),
        F.col(f"data.{road}.flowSegmentData.freeFlowSpeed").alias("free_flow_speed"),
        F.col(f"data.{road}.flowSegmentData.freeFlowTravelTime").alias("free_flow_travel_time"),
        F.col(f"data.{road}.flowSegmentData.frc").alias("frc"),
        F.col(f"data.{road}.flowSegmentData.roadClosure").alias("road_closure"),
        F.col(f"data.{road}.flowSegmentData.confidence").alias("confidence"),
        F.col(f"data.{road}.flowSegmentData.coordinates.coordinate").alias("coordinates"),
        F.col("_ingestion_timestamp"),
        F.col("_source_file")
    )

In [0]:
tomtom_flatten = flatten_road(bronze_tomtom_flow, roads[0])
for road in roads[1:]:
    tomtom_flatten = tomtom_flatten.unionByName(
        flatten_road(bronze_tomtom_flow, road)
    )

In [0]:
tomtom_types = tomtom_flatten\
    .withColumn("measurement_timestamp", F.to_timestamp("fetched_at"))\
    .withColumn("current_speed", F.col("current_speed").cast("double"))\
    .withColumn("current_travel_time", F.col("current_travel_time").cast("integer"))\
    .withColumn("free_flow_speed", F.col("free_flow_speed").cast("double"))\
    .withColumn("free_flow_travel_time", F.col("free_flow_travel_time").cast("integer"))\
    .withColumn("confidence",F.col("confidence").cast("double"))\
    .drop(F.col("fetched_at"))

tomtom_types.printSchema()


### Dedup

In [0]:
tomtom_dedup = tomtom_types.dropDuplicates(["road_name","measurement_timestamp"])


### Valid

In [0]:
tomtom_valid = tomtom_dedup.filter(
    (F.col("road_name").isNotNull()) &
    (F.col("measurement_timestamp").isNotNull()) &
    (F.col("current_speed").isNotNull()) &
    (F.col("current_speed") >= 0) &
    (F.col("current_travel_time") >= 0) &
    (F.col("free_flow_speed") >= 0) &
    (F.col("free_flow_travel_time") >= 0) &
    (F.col("confidence").between(0,1))

).withColumn("municipality_name", find_municipality_udf(F.col('coordinates')[0]['latitude'], F.col('coordinates')[0]['longitude']))



### Merge

In [0]:
def merge_tomtom_flow(df_source, batch_id):
    silver_table_tomtom_flow = DeltaTable.forName(
        spark,
        "bg_traffic.bg_traffic_silver.tomtom_flow"
    ).alias("target").merge(df_source.alias("source"), """
                                target.road_name = source.road_name
                                AND target.measurement_timestamp = source.measurement_timestamp
                            """)\
                                .whenMatchedUpdateAll()\
                                    .whenNotMatchedInsertAll()\
                                        .execute()

In [0]:
query = (
    tomtom_valid
    .writeStream
    .foreachBatch(merge_tomtom_flow)
    .option("checkpointLocation", tomtom_flow_checkpoint)
    .trigger(availableNow = True)
    .start()
)

query.awaitTermination()

In [0]:
%sql
SELECT * FROM bg_traffic.bg_traffic_silver.tomtom_flow;